# 07 · Preprocesamiento v3 (S0)
**Proyecto PLN · Entrega 3** — Pregunta guía: *¿qué tan estable es la agenda temática de la prensa colombiana entre 1980 y 2011 y cuándo cambia?*

Pipeline único y cacheado que consumen los notebooks 08 (tópicos) y 09 (cronolecto). Resuelve:

1. Artefactos de maquetación y scraping (fotos, URLs, VER FOTO/TABLA/GRÁFICO, firmas de agencia → columna `agencia`).
2. Titular separado del cuerpo (`titulo`, `cuerpo`), con los titulares en Mayúscula Inicial normalizados antes de spaCy.
3. NER de spaCy sobre el cuerpo, en lugar del proxy de mayúsculas del 05/06.
4. Géneros que no son noticia: `es_carta`, `es_breve` (se marcan, no se borran).
5. NPMI corregido sobre un único flujo de tokens, con assert de rango.
6. Una sola tokenización (spaCy) para todo el proyecto.

Código: lo liviano en `utils.py` (solo agregados retrocompatibles, ver `tests/test_regresion_utils.py`) y lo pesado en `prep.py`. Salida: `data/corpus_procesado.parquet` y `data/bigramas_npmi.csv`.

In [ ]:
import os
import time
from collections import Counter

import pandas as pd
import prep
import utils

utils.setup()
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

## 0. Configuración
Primero `FAST = True` (6.000 documentos al azar, `seed=42`) para probar el pipeline; luego `FAST = False` para la corrida completa. Cada modo escribe sus propios archivos: en FAST, el parquet, los bigramas, los candidatos y el diagnóstico llevan el sufijo `_fast`, así que nunca pisan los del corpus completo. Desde la terminal, `PREP_FAST=1` fuerza el modo FAST.

El caché se invalida solo si cambia algo que afecta la salida: la configuración, `data/stopwords_dominio.txt`, `PIPELINE_VERSION`, o la versión de spaCy o del modelo. `n_process` y `batch_size` no entran al hash. `umbral_breve` va en un hash aparte, así que cambiarlo recalcula las marcas sin volver a correr spaCy.

In [ ]:
FAST = False  # por defecto: corpus completo
SUF = "_fast" if FAST else ""
N_PROCESS = max(1, min(8, (os.cpu_count() or 2) - 2))  # 1-1,5 GB de RAM por proceso

cfg = prep.PrepConfig(fast=FAST, n_process=N_PROCESS, batch_size=64)
print(cfg)
print("hash:", prep.config_hash(cfg))

## 1. Separador del titular (verificación empírica)
Se mide qué separador candidato aparece en los documentos y cuánto mide el primer segmento.

In [ ]:
df = utils.load_corpus()
df["año"] = df["fecha"].dt.year
txt = utils.normalizar_texto(df["texto"])

candidatos = {
    "salto doble \\n\\n": "\n\n",
    "salto simple \\n": "\n",
    "doble espacio": "  ",
}
print(
    pd.Series(
        {
            k: txt.str.contains(v, regex=False).mean() * 100
            for k, v in candidatos.items()
        }
    )
    .round(2)
    .rename("% documentos")
    .to_string()
)
primer = txt.str.split("\n\n", n=1).str[0]
print("\nPalabras del primer segmento (antes del primer \\n\\n):")
print(primer.str.split().str.len().describe().round(1).to_string())

**Decisión:** el separador es el primer salto doble `\n\n`. Aparece en el 100 % de los documentos y el primer segmento tiene una mediana de 6 palabras, que es el tamaño de un titular. El doble espacio no sirve como separador: no aparece en el texto crudo, y el ~7 % que se ve aquí son los espacios que deja `normalizar_texto` al reemplazar caracteres de control. `separar_titulo` lo descarta como titular si supera 40 palabras.

El titular en Mayúscula Inicial no es solo de El Tiempo 2000–2005: va de **1990 a 2005 en El Tiempo** y de **1982 a 1999 en Semana** (tabla de abajo). Con 4 o más palabras de 4+ letras, se marca `titulo_capitalizado` si al menos el 80 % empiezan en mayúscula. Antes de spaCy esos titulares se pasan a mayúscula solo en la primera letra. Así el etiquetador no toma cada palabra por nombre propio: en "Un Bus Ejecutivo…" etiquetaba `Bus`→PROPN, y en minúscula `bus`→NOUN.

In [ ]:
base = prep.preparar_textos(
    df, cfg
)  # limpieza completa sobre los 91.585 docs (sin spaCy)

det = base.assign(detectado=base["sep_titulo"].notna())
print("% titulares detectados por fuente:")
print(det.groupby("fuente")["detectado"].mean().mul(100).round(2).to_string())
print("\n% titulares en Mayúscula Inicial por año y fuente:")
print(
    base.pivot_table(
        index="año", columns="fuente", values="titulo_capitalizado", aggfunc="mean"
    )
    .mul(100)
    .round(0)
    .to_string()
)

## 2. Limpieza de artefactos
`utils.limpiar_artefactos` se aplica por separado al titular y al cuerpo, después de `normalizar_texto`. El corpus trae la puntuación separada por espacios (`www . semana . com`) y las regex lo tienen en cuenta:

| Artefacto | Ejemplo | Regla |
|---|---|---|
| HTML | `< img src = ... >` | etiqueta completa |
| Correo / URL | `www . icetex . gov . co`, `Semana . com` | dominio con TLD; prefiltro para no recorrer todo el corpus |
| Crédito de foto | `FOTO / Felipe Caicedo EL TIEMPO .`, `EFE / Raúl García` | desde `FOTO /` hasta ` . `, ` - ` o fin de línea |
| Leyenda | `Foto : .`, `Fotoleyenda :`, `Pie de foto`, `FOTO No . 2` | prefijo de línea |
| Remisión | `( VER FOTO )`, `( VER : EN CÚCUTA … )`, `( en la foto )` | RE_MAQ del 02 + RE_VER |
| Agencia | `( AFP - Efe )`, `... EFE` al final, `MONTEVIDEO AFP El …` | se quita y se guarda en `agencia` (EFE, AFP, AP, REUTERS) |

Las menciones en prosa ("dijo a la AP en el hotel", "las AFP" de pensiones) **no** se tocan.

Además: **lead duplicado.** En El Tiempo 2001–2005 y en Semana 2000–2003, entre el 15 % y el 25 % de los documentos repiten el primer párrafo (artefacto del scraping). Eso duplica las frecuencias justo en el periodo que se quiere comparar. Se elimina la primera copia (`quitar_lead_duplicado=True`) y queda marcado en `lead_duplicado`.

In [ ]:
import re

cuerpo_limpio = base["titulo"] + "\n\n" + base["cuerpo"]
# Regex de Python (no RE2 de pyarrow): en RE2, \b solo reconoce letras ASCII y
# "común" contaría como "com".
PATRONES = {
    "foto (palabra)": re.compile(r"\bfotos?\b", re.IGNORECASE),
    "crédito FOTO /": re.compile(r"\b(?:FOTO|Foto)S? */"),
    "www": re.compile(r"\bwww\b", re.IGNORECASE),
    "com": re.compile(r"\bcom\b", re.IGNORECASE),
    "VER FOTO/TABLA/GRÁFICO": re.compile(
        r"\(\s*VER\b|VER\s+(?:FOTO|TABLA|GRAFICO|GRÁFICO)", re.IGNORECASE
    ),
    "efe": re.compile(r"\befe\b", re.IGNORECASE),
    "ap": re.compile(r"\bAP\b"),
    "afp": re.compile(r"\bafp\b", re.IGNORECASE),
}


def n_docs(serie, rx):
    return int(serie.map(lambda t: rx.search(t) is not None).sum())


art = pd.DataFrame(
    {
        "docs_antes": {k: n_docs(txt, rx) for k, rx in PATRONES.items()},
        "docs_después": {k: n_docs(cuerpo_limpio, rx) for k, rx in PATRONES.items()},
    }
)
art["% quitado"] = (1 - art["docs_después"] / art["docs_antes"]).mul(100).round(1)
print(art.to_string())
print("\nLo que queda de 'foto', 'efe', 'afp' es prosa ('una foto de', 'dijo a Efe').")
print("\nAgencia detectada:")
print(base["agencia"].value_counts(dropna=False).to_string())
print("\nLead duplicado (%) por año y fuente:")
print(
    base.pivot_table(
        index="año", columns="fuente", values="lead_duplicado", aggfunc="mean"
    )
    .mul(100)
    .round(1)
    .loc[1999:]
    .to_string()
)

## 3. spaCy, stopwords y construcción del parquet
- `es_core_news_md` **sin parser**; quedan `tok2vec`, `morphologizer` (POS; este modelo no trae un `tagger` aparte), `attribute_ruler`, `lemmatizer` y `ner`. `nlp.pipe(n_process, batch_size=64)`.
- **Una sola tokenización.** `tokens_forma` (minúsculas, sin puntuación ni símbolos, con stopwords) es la base. De la misma pasada salen `lemas_contenido` (NOUN, ADJ, VERB, PROPN; alfabéticos, ≥3 letras, sin stopwords por forma ni por lema) y `lemas_nom_adj` (solo NOUN y ADJ, para tópicos).
- `pos_contenido`: una letra por lema (N/A/V/P). `cortes_contenido`: posiciones de `lemas_contenido` donde había puntuación, salto de línea, frontera titular/cuerpo o más de 3 tokens descartados seguidos. El NPMI no cuenta pares que crucen un corte.
- **NER solo del cuerpo**: se descartan las entidades que empiezan dentro del titular y se recortan en el primer salto de línea (sin parser, la NER a veces cruza párrafos).
- Rachas de 3+ palabras en mayúscula sostenida (versalitas de Semana: "A PESAR DE QUE EN SU FAMILIA…") pasan a minúscula **solo** en el texto que entra a spaCy. Así se evitan entidades falsas como `SU FAMILIA NO`→MISC.
- **Stopwords**: NLTK + spaCy + `data/stopwords_dominio.txt`, con secciones auditables (abajo). Dos secciones rescatan palabras que las listas de NLTK y spaCy marcan como stopword:
  - `conservar` (**estado**, **poder**, **final**, **verdad**): se conservan cuando el token es sustantivo **o nombre propio** ("el estado", "el Estado", "Poder Judicial").
  - `conservar_propn`: se conservan **solo** cuando el token es nombre propio, porque son la primera palabra de un nombre compuesto (**estados** → Estados Unidos, **nueva** → Nueva York, **medio** → Medio Ambiente, **buenos** → Buenos Aires…). Como sustantivo o adjetivo siguen siendo stopword ("medio de comunicación", "una nueva ley").
- **Sustantivos que solo spaCy marca como stopword (versión 3.0.3).** La lista de spaCy incluye palabras de contenido (acuerdo, medio, fin, cuenta, uso, bienes, hechos, embargo). Hasta la 3.0.2 se descartaban también como sustantivo: "acuerdo de paz" quedaba en "paz". Ahora están en `conservar`, pero con un filtro: si el NOUN cae en una locución gramatical (`prep.LOCUCIONES`: "de acuerdo con", "con el fin de", "por medio de", "en cuenta", "sin embargo", "de hecho"…), se descarta igual. Así entra "el acuerdo humanitario" y no entra "de acuerdo con los datos". La evidencia está en el diagnóstico 3.2.
- **Por qué (versión 3.0.2).** Hasta la 3.0.1, `conservar` solo aplicaba a NOUN. spaCy etiqueta "Estados Unidos" como PROPN, y "estados" está en NLTK y spaCy, así que se descartaba y quedaba solo "unidos". Eso producía bigramas falsos como `gobierno_unidos` o `presidente_unidos`, y `estados_unidos` tenía n = 22. El diagnóstico 3.1 muestra qué otros nombres propios se perdían por la misma razón. La verificación 4.8 comprueba la corrección.

In [ ]:
secciones = utils.leer_stopwords_dominio()
print({k: len(v) for k, v in secciones.items()})
print("Total stopwords (NLTK + spaCy + dominio):", len(utils.cargar_stopwords()))
print("nombres_pila (muestra):", secciones["nombres_pila"][:25])

### 3.1 Diagnóstico: nombres propios que se pierden por las stopwords
Se toma una muestra FAST (6.000 docs, `seed=42`), se pasa por spaCy y se listan los tokens PROPN cuyo lema o forma está en las stopwords. Son los que la regla de la 3.0.1 descartaba. Para cada uno se muestra de qué lista viene, la palabra siguiente más frecuente (la entidad que se rompe) y un ejemplo.

- **Se rescatan** (`conservar_propn`) solo los que encabezan nombres compuestos frecuentes: Estados Unidos, Nueva York, Medio Ambiente, Buenos Aires, Alcaldía Mayor, Once Caldas, Primera Dama, Segunda Guerra, Primer Ministro, Grandes Ligas, Bajo Cauca y Santo Domingo. "estado" y "poder" PROPN ("el Estado", "Poder Judicial") quedan rescatados porque `conservar` ahora aplica también a PROPN.
- **No se rescatan:**
  - nombres de pila y meses (se quitan a propósito; la columna `entidades` conserva el nombre completo);
  - metatexto ("Señor Director", "Semana" como nombre de la revista);
  - residuos de scraping;
  - palabras funcionales que spaCy etiqueta PROPN por error al inicio de oración ("Este es…", "Uno", "Primero ,").

La columna `rescatado` indica si la regla actual (3.0.2) lo conserva.

In [ ]:
muestra_diag = base.sample(n=min(cfg.n_fast, len(base)), random_state=cfg.seed)
t0 = time.time()
diag = prep.diagnostico_propn_perdidos(muestra_diag, cfg, top=2000)
print(
    f"{time.time() - t0:.0f} s | grupos (lema, forma) PROPN en stopwords: {len(diag)}"
)
print("\nTokens por lista de origen:")
print(
    diag.groupby("origen")["n"].sum().sort_values(ascending=False).head(10).to_string()
)

intencional = diag["origen"].str.fullmatch(r"dominio:(nombres_pila|meses)")
print(
    f"\nNombres de pila y meses (descartados a propósito): {int(diag.loc[intencional, 'n'].sum()):,} tokens"
)
cols_d = [
    "lema",
    "forma",
    "n",
    "docs",
    "origen",
    "siguiente_top",
    "rescatado",
    "ejemplo",
]
print("Top-40 restante:")
diag[~intencional].head(40)[cols_d]

In [ ]:
t0 = time.time()
dfp = prep.construir_corpus_procesado(cfg)
meta = dfp.attrs["prep_meta"]
print(
    f"\n{dfp.shape[0]:,} documentos × {dfp.shape[1]} columnas | celda: {time.time() - t0:.0f} s"
)
dfp.head(3)

### 3.2 Diagnóstico: sustantivos que solo spaCy marca como stopword
Se calcula durante la construcción del parquet, en la misma pasada de spaCy, sin costo extra. Se guarda en `data/diag_noun_spacy{SUF}.csv`: en FAST sobre la muestra y en la corrida completa sobre los 91.585 documentos.

Se incluyen los tokens cuyo lema o forma está en `STOP_WORDS` de spaCy y **no** en NLTK ni en la lista de dominio. Columnas:
- `n_noun`, `df`: apariciones y documentos como NOUN.
- `frac_noun`: fracción de sus apariciones que son NOUN (el resto, otros POS).
- `frac_locucion`: fracción de las apariciones NOUN que caen en una locución gramatical de `prep.LOCUCIONES`.
- `n_fuera_locucion`: lo que entra a `lemas_nom_adj` si el lema está en `conservar`.
- `ejemplo_1`, `ejemplo_2`: contextos fuera de locución. `ejemplo_locucion`: uno dentro.

**Decisión (con la muestra FAST):**
- **Se rescatan:** acuerdo, medio, fin, cuenta, uso, bien (bienes), hecho (hechos) y embargo. Fuera de locución son temáticos: "acuerdo humanitario", "medio ambiente", "medios de comunicación", "fin del conflicto", "cuentas bancarias", "rendición de cuentas", "uso del suelo", "bienes y servicios", "esclarecer los hechos", "embargo de bienes".
- **No se rescatan:**
  - parte: "hacer parte de", "buena parte de", "por parte de"; el uso temático ("las partes" de una negociación) no se separa con fiabilidad.
  - tarde y total: la mayoría son locuciones temporales o de cantidad ("en la tarde", "un total de").
  - lado: "por otro lado", "al lado de".
  - El resto de la lista son usos gramaticales mal etiquetados como NOUN ("a través de", "a pesar de", "a partir de", "manera", "respecto", "veces").

In [ ]:
diag_noun = pd.read_csv(prep.ruta_diag_noun(cfg))
print(prep.ruta_diag_noun(cfg), "| lemas:", len(diag_noun))
cols_n = [
    "lema",
    "n_noun",
    "df",
    "frac_noun",
    "frac_locucion",
    "n_fuera_locucion",
    "rescatado",
    "ejemplo_1",
    "ejemplo_2",
    "ejemplo_locucion",
]
diag_noun.head(40)[cols_n]

## 4. Verificaciones (criterios de aceptación)
### 4.1 Top-200 de lemas
Hay dos pruebas. La oficial es sobre `lemas_contenido`. La estricta recalcula el ranking sobre `tokens_forma` **sin** la sección `scraping` de stopwords. Así se demuestra que esos residuos desaparecieron por la limpieza y no porque estén en una lista.

In [ ]:
PROHIBIDOS = ["foto", "com", "www", "efe", "ap", "afp"]

c = Counter()
for x in dfp["lemas_contenido"]:
    c.update(x)
top200 = [w for w, _ in c.most_common(200)]
assert not set(PROHIBIDOS) & set(top200), set(PROHIBIDOS) & set(top200)
assert not [w for w in c if " " in w], "hay lemas con espacio"
print("OK: ninguno en el top-200 de lemas_contenido")
print("Top-60:", " ".join(top200[:60]))

sw_sin_scraping = utils.cargar_stopwords(excluir_secciones=("scraping",))
cf = Counter()
for x in dfp["tokens_forma"]:
    cf.update(t for t in x if t.isalpha() and t not in sw_sin_scraping)
rango = {w: i + 1 for i, (w, _) in enumerate(cf.most_common())}
estricta = pd.DataFrame(
    {
        "frecuencia": [cf[w] for w in PROHIBIDOS],
        "rango": [rango.get(w) for w in PROHIBIDOS],
    },
    index=PROHIBIDOS,
)
print(estricta.to_string())
assert estricta["rango"].fillna(10**9).gt(200).all()
print("OK prueba estricta: fuera del top-200 aun sin la sección scraping")

### 4.2 Cinco documentos antes y después

In [ ]:
raw = df.set_index("id")["texto"]


def elegir(mask, etiqueta):
    sub = dfp[mask]
    return (
        (etiqueta, int(sub.sample(1, random_state=1)["id"].iloc[0]))
        if len(sub)
        else None
    )


et = dfp["fuente"].str.contains("eltiempo")
ejemplos = [
    e
    for e in [
        elegir(
            et & (dfp["año"] == 2002) & dfp["titulo_capitalizado"],
            "El Tiempo 2002, titular en Mayúscula Inicial",
        ),
        elegir(dfp["agencia"].notna() & (dfp["año"] >= 2006), "Firma de agencia"),
        elegir(
            raw.reindex(dfp["id"])
            .str[:450]
            .str.contains(r"\b(?:FOTO|Foto) */", regex=True)
            .values,
            "Crédito de foto",
        ),
        elegir(dfp["lead_duplicado"], "Lead duplicado"),
        elegir(dfp["es_carta"], "Carta al director"),
    ]
    if e
]


def mostrar(etiqueta, id_):
    r = dfp.set_index("id").loc[id_]
    print("=" * 110)
    print(
        f"[{etiqueta}] id={id_} | {r['fuente']} {r['fecha']:%Y-%m-%d} | agencia={r['agencia']} | "
        f"carta={r['es_carta']} breve={r['es_breve']} ({r['breve_motivo']}) | lead_dup={r['lead_duplicado']}"
    )
    print("--- ANTES (inicio … final) ---")
    print(raw[id_][:450].rstrip(), "\n   […]\n", raw[id_].rstrip()[-200:])
    print("--- DESPUÉS ---")
    print("TÍTULO:", r["titulo"], "| capitalizado:", r["titulo_capitalizado"])
    print("CUERPO:", r["cuerpo"][:450].rstrip(), "\n   […]\n", r["cuerpo"][-200:])
    print("lemas_contenido:", " ".join(r["lemas_contenido"][:25]))
    print("lemas_nom_adj:  ", " ".join(r["lemas_nom_adj"][:25]))
    print("entidades:", [(e["texto"], e["tipo"]) for e in r["entidades"][:10]])


for etiqueta, id_ in ejemplos:
    mostrar(etiqueta, id_)

### 4.3 Titulares detectados por fuente y año

In [ ]:
print("% detectado (sep_titulo no nulo):")
print(
    dfp.assign(det=dfp["sep_titulo"].notna())
    .pivot_table(index="año", columns="fuente", values="det", aggfunc="mean")
    .mul(100)
    .round(1)
    .to_string()
)
print("\n% en Mayúscula Inicial:")
print(
    dfp.pivot_table(
        index="año", columns="fuente", values="titulo_capitalizado", aggfunc="mean"
    )
    .mul(100)
    .round(0)
    .to_string()
)

### 4.4 `es_breve` y `es_carta`
**Criterio de `es_breve`**: se cumple una de dos condiciones.
- **nota**: el titular termina en `:`. Es la columna de breves de El Tiempo entre 1990 y 2005 (`Título :\n\n. texto`). Esos documentos tienen una mediana de 68 palabras de cuerpo, frente a 375 en el resto.
- **corto**: el cuerpo tiene menos de `umbral_breve` palabras (80 por defecto), sin contar el titular.

`breve_motivo` indica cuál de las dos se cumplió. Como `n_palabras_cuerpo` queda guardado, en 08 y 09 se puede usar otro umbral sin volver a correr nada. La tabla de sensibilidad muestra cómo cambia la marca con 50, 80, 100 y 150 palabras.

`es_carta`: "señor director" en el titular o el cuerpo.

In [ ]:
def pct(col):
    return (
        dfp.pivot_table(index="año", columns="fuente", values=col, aggfunc="mean")
        .mul(100)
        .round(1)
    )


print("es_breve (%):")
print(pct("es_breve").to_string())
print("\nbreve_motivo:", dfp["breve_motivo"].value_counts(dropna=False).to_dict())
print("\nes_carta (n):")
print(
    dfp.pivot_table(
        index="año", columns="fuente", values="es_carta", aggfunc="sum"
    ).to_string()
)
print("total cartas:", int(dfp["es_carta"].sum()))

sens = pd.DataFrame(
    {
        f"<{u}": dfp.assign(c=dfp["n_palabras_cuerpo"] < u)
        .query("fuente == 'www.eltiempo.com'")
        .groupby("año")["c"]
        .mean()
        .mul(100)
        .round(1)
        for u in (50, 80, 100, 150)
    }
)
print("\nSensibilidad del umbral 'corto' — El Tiempo, % de documentos por año:")
print(sens.to_string())

### 4.5 Colocaciones NPMI corregidas
**Error del 05:** `p(ab)` se normalizaba por el total de bigramas y `p(a)` por `N_CONT`, el total de tokens de contenido. Son dos poblaciones distintas, y por eso salían valores mayores que 1.

**Corrección:** un solo flujo de tokens, `lemas_contenido` con sus cortes. `p(ab) = n(a,b)/N`, y `p(a)`, `p(b)` son las marginales izquierda y derecha **de la misma tabla** de bigramas. Como `n(a,b)` nunca supera su marginal, NPMI queda en [−1, 1] por construcción. Igual se verifica con `assert` sobre todos los pares.

**Por qué lemas con cortes.** Primero se comparan las tres variantes. En el flujo de lemas, los bigramas quedan en el mismo espacio que los features de tópicos de S1 (`ministro_defensa`) y las variantes flexivas suman sus frecuencias. Los cortes evitan pares que no existen en el texto, como "Federación de Comerciantes ( Fenalco )" → `comerciantes_fenalco`.

In [ ]:
sw = utils.cargar_stopwords()
muestra = dfp.sample(min(len(dfp), 6000), random_state=42).reset_index(drop=True)
resumen, lado = prep.comparar_variantes_npmi(muestra, sw, min_n=5, min_df=3, k=300)
print(resumen.to_string())
lado

**Umbrales.**
- `min_n = 20`: el NPMI sobreestima los pares raros, y con 20 ocurrencias el error relativo del conteo (≈1/√n) baja a ~22 %.
- `min_df = 10` documentos: evita que gane un par repetido dentro de unos pocos documentos (tablas de bolsa, resultados deportivos).
- `NPMI ≥ 0,35`: se justifica con los cuantiles y con la revisión manual de la franja 0,30–0,40 (abajo).
- En FAST se usan 5 y 3, porque la muestra es ~15 veces más chica.

Para S1 se exportan todos los candidatos con su `tipo`:
- `nom_adj`: sustantivo/adjetivo con sustantivo/adjetivo. Son los features de tópicos.
- `con_propn`: incluye nombres propios. Se marcan aparte, porque en el 05 las entidades dominaban los cambios.
- `con_verbo`: incluye un verbo.

In [ ]:
MIN_N, MIN_DF = (5, 3) if FAST else (20, 10)
UMBRAL_NPMI = 0.35

t0 = time.time()
bg = prep.colocaciones_npmi(
    dfp["lemas_contenido"],
    dfp["cortes_contenido"],
    dfp["pos_contenido"],
    min_n=MIN_N,
    min_df=MIN_DF,
)
print(
    f"{time.time() - t0:.0f} s | N bigramas = {bg.attrs['N']:,} | pares distintos = {bg.attrs['n_pares']:,}"
)
print("Rango NPMI en TODOS los pares:", bg.attrs["rango_todos"])
assert bg["npmi"].between(-1, 1).all()
print("OK rango [−1, 1]")
print(f"\nCandidatos con n≥{MIN_N}, df≥{MIN_DF}: {len(bg):,}")
print(bg["tipo"].value_counts().to_string())
print("\nCuantiles de NPMI:")
print(bg["npmi"].quantile([0.5, 0.75, 0.9, 0.95, 0.99]).round(3).to_string())

In [ ]:
cols = ["w1", "w2", "n", "df", "npmi", "tipo"]
print("Top-30 general (dominan nombres propios):")
print(bg.head(30)[cols].round(3).to_string())
print("\nTop-30 nom_adj (features de tópicos):")
print(bg[bg["tipo"] == "nom_adj"].head(30)[cols].round(3).to_string())

In [ ]:
na = bg[bg["tipo"] == "nom_adj"]
print(
    "Candidatos nom_adj por umbral:",
    {u: int((na["npmi"] >= u).sum()) for u in (0.25, 0.30, 0.35, 0.40, 0.50)},
)
franja = na[(na["npmi"] >= 0.30) & (na["npmi"] < 0.40)]
print("\nFranja 0,30–0,40 (muestra para revisión manual):")
print(
    franja.sample(min(30, len(franja)), random_state=0)
    .sort_values("npmi")[cols]
    .round(3)
    .to_string()
)

In [ ]:
ruta_bg = prep.exportar_bigramas(
    bg, umbral=UMBRAL_NPMI, path=utils.data_dir() / f"bigramas_npmi{SUF}.csv"
)
exp = pd.read_csv(ruta_bg)
print(f"{ruta_bg} | {len(exp):,} bigramas (npmi ≥ {UMBRAL_NPMI})")
print(exp["tipo"].value_counts().to_string())

### 4.6 Tiempo de ejecución y tamaño

In [ ]:
meta = prep.leer_meta(prep.ruta_salida(cfg))
ruta = prep.ruta_salida(cfg)
print(f"Parquet: {ruta} — {ruta.stat().st_size / 2**20:.1f} MB")
print(
    f"Tiempo de construcción: {meta.get('segundos_total')} s "
    f"(limpieza {meta.get('segundos_limpieza')} s) | n_process={meta.get('n_process')} batch_size={meta.get('batch_size')}"
)
print(
    f"Documentos: {meta.get('n_docs'):,} | hash nlp {meta['hash_nlp']} | hash géneros {meta['hash_generos']}"
)
print(
    dfp[
        [
            "n_palabras_titulo",
            "n_palabras_cuerpo",
            "n_lemas_contenido",
            "n_lemas_nom_adj",
            "n_entidades",
        ]
    ]
    .describe()
    .round(1)
    .to_string()
)

### 4.7 Candidatos a lista blanca `con_propn` (para el 08)
Los bigramas con nombre propio quedan fuera de los features de tópicos, pero algunos son instituciones que sí son tema (`estados_unidos`, `fuerzas_militares`). Aquí se exportan los candidatos, sin umbral de NPMI y ordenados por frecuencia documental, con una marca heurística de "institucional". La lista final se elige a mano y se guarda en `data/lista_blanca_con_propn.txt`.

In [ ]:
INST = re.compile(
    r"^(corte|consejo|congreso|senado|cámara|ministerio|fuerza|fuerzas|ejército|"
    r"armada|policía|fiscalía|procuraduría|contraloría|defensoría|registraduría|"
    r"superintendencia|banco|naciones|unión|estados|comisión|tribunal|asamblea|"
    r"partido|federación|organización|fondo|instituto|alcaldía|gobernación|junta|"
    r"frente|autodefensas|reserva|bolsa|agencia|unidos|unidas|nacional|militares|"
    r"armadas|europea|constitucional|suprema|república|general|electoral|monetario)$"
)
cp = bg[bg["tipo"] == "con_propn"].copy()
cp.insert(0, "bigrama", cp["w1"] + "_" + cp["w2"])
cp["institucional"] = cp["w1"].str.match(INST) | cp["w2"].str.match(INST)
cp["en_csv_035"] = cp["npmi"] >= UMBRAL_NPMI
cp = cp.sort_values("df", ascending=False).reset_index(drop=True)

ruta_cand = utils.data_dir() / f"candidatos_lista_blanca_con_propn{SUF}.csv"
cp.head(500)[
    ["bigrama", "w1", "w2", "n", "df", "npmi", "pos", "institucional", "en_csv_035"]
].round({"npmi": 4}).to_csv(ruta_cand, index=False)

cols_c = ["bigrama", "n", "df", "npmi", "en_csv_035"]
print(
    f"{ruta_cand} | con_propn: {len(cp):,} | institucionales: {int(cp['institucional'].sum())}"
)
print("\nInstitucionales (top 80 por df):")
print(cp[cp["institucional"]].head(80)[cols_c].round(3).to_string(index=False))
print("\nNo marcadas, top 40 por df (por si la regex se perdió alguna):")
print(cp[~cp["institucional"]].head(40)[cols_c].round(3).to_string(index=False))

In [ ]:
cu, prev, ce = Counter(), Counter(), Counter()
for lem, pos in zip(dfp["lemas_contenido"], dfp["pos_contenido"]):
    for i, w in enumerate(lem):
        if w == "unidos":
            cu[pos[i]] += 1
            prev[lem[i - 1] if i else "<ini>"] += 1
        elif w in ("estado", "estados"):
            ce[(w, pos[i])] += 1
n_eeuu = sum(" ".join(t).count("estados unidos") for t in dfp["tokens_forma"])
print("'estados unidos' en tokens_forma:", n_eeuu)
print("'unidos' en lemas_contenido por POS:", dict(cu))
print("estado/estados en lemas_contenido por POS:", dict(ce))
print("lema anterior a 'unidos' (top 15):", prev.most_common(15))
print(
    bg[(bg["w1"].isin(["estado", "estados"])) & (bg["w2"].isin(["unido", "unidos"]))][
        ["w1", "w2", "n", "df", "npmi", "tipo"]
    ].to_string()
)

### 4.8 Verificación: nombres propios rescatados (3.0.2)
Criterios:
1. "Estados" se conserva como PROPN (lema `estados`, POS `P`), en una cantidad del orden de las apariciones de "estados unidos" en `tokens_forma`. spaCy no lematiza "Estados" cuando lo etiqueta PROPN, por eso el lema es `estados` y no `estado`. "el Estado" sí queda como `estado`/P.
2. El bigrama `estados_unidos` aparece en la tabla NPMI como `con_propn`, con n del orden de esas mismas apariciones.
3. Los bigramas falsos `gobierno_unidos` y `presidente_unidos` desaparecen: en la 3.0.1 eran el 2,2 % y el 1,9 % de los "unidos" (405 y 344 en el corpus completo).
4. Esquema y número de documentos iguales a la 3.0.1, y la prueba del top-200 (4.1) sigue pasando.

In [ ]:
n_eeuu = sum(" ".join(t).count("estados unidos") for t in dfp["tokens_forma"])
lema_pos, previo = Counter(), Counter()
for lem, pos in zip(dfp["lemas_contenido"], dfp["pos_contenido"]):
    for i, w in enumerate(lem):
        lema_pos[(w, pos[i])] += 1
        if w == "unidos":
            previo[lem[i - 1] if i else "<ini>"] += 1
n_unidos = sum(previo.values())

secc = utils.leer_stopwords_dominio()
rescates = sorted(set(secc["conservar"]) | set(secc["conservar_propn"]))
print("'estados unidos' en tokens_forma:", n_eeuu)
print("Rescatados como PROPN (lema, n en lemas_contenido):")
print({w: lema_pos[(w, "P")] for w in rescates})
print("Lema anterior a 'unidos' (top 8):", previo.most_common(8))

# 1. "Estados" como PROPN, del orden de "estados unidos"
assert 0.8 * n_eeuu <= lema_pos[("estados", "P")] <= 1.5 * n_eeuu, lema_pos[
    ("estados", "P")
]
assert lema_pos[("estado", "P")] > 0  # "el Estado"

# 2. Bigrama estados_unidos
fila = bg[(bg["w1"] == "estados") & (bg["w2"] == "unidos")]
print(fila[["w1", "w2", "n", "df", "npmi", "tipo"]].to_string())
assert len(fila) == 1 and fila["tipo"].iloc[0] == "con_propn"
assert fila["n"].iloc[0] >= 0.8 * n_eeuu

# 3. Bigramas falsos: "unidos" ahora va precedido casi siempre de "estados"
assert previo["estados"] / n_unidos >= 0.9, previo["estados"] / n_unidos
for w in ("gobierno", "presidente"):
    assert previo[w] / n_unidos < 0.002, (w, previo[w])

# 4. Esquema, documentos y top-200
assert list(dfp.columns) == prep.SCHEMA.names
assert FAST or len(dfp) == utils.N_DOCS_UNICOS
assert not set(PROHIBIDOS) & set(top200)
print(
    "OK 4.8: Estados Unidos rescatado, bigramas falsos eliminados, esquema y top-200 intactos"
)

### 4.9 Verificación: sustantivos rescatados (3.0.3)
1. "acuerdo" aparece en `lemas_nom_adj`.
2. Para cada lema rescatado, las apariciones en `lemas_nom_adj` coinciden (±0,5 %) con `n_fuera_locucion` del diagnóstico 3.2. Así se comprueba que entra exactamente lo que el diagnóstico considera temático.
3. `acuerdo_paz` y `acuerdo_humanitario` están en la tabla NPMI (n ≥ MIN_N, df ≥ MIN_DF). Se indica si pasan el umbral de exportación.
4. La prueba del top-200 (4.1) sigue pasando; mismo esquema y, en la corrida completa, 91.585 documentos.

In [ ]:
RESCATE_303 = ["acuerdo", "medio", "fin", "cuenta", "uso", "bien", "hecho", "embargo"]
assert set(RESCATE_303) <= set(utils.leer_stopwords_dominio()["conservar"])

en_na = Counter()
for x in dfp["lemas_nom_adj"]:
    en_na.update(w for w in x if w in RESCATE_303)
dn = diag_noun.set_index("lema")
tabla_303 = pd.DataFrame(
    {
        "en_lemas_nom_adj": {w: en_na[w] for w in RESCATE_303},
        "n_fuera_locucion": {
            w: int(dn.loc[w, "n_fuera_locucion"]) for w in RESCATE_303
        },
        "n_noun_total": {w: int(dn.loc[w, "n_noun"]) for w in RESCATE_303},
        "frac_locucion": {w: dn.loc[w, "frac_locucion"] for w in RESCATE_303},
    }
)
print(tabla_303.to_string())
print(f"Tokens nuevos en lemas_nom_adj: {sum(en_na.values()):,}")

# 1. acuerdo en lemas_nom_adj
assert en_na["acuerdo"] > 0
# 2. entra lo que el diagnóstico marca como fuera de locución
for w in RESCATE_303:
    a, b = en_na[w], tabla_303.loc[w, "n_fuera_locucion"]
    assert abs(a - b) <= max(2, 0.005 * b), (w, a, b)

# 3. bigramas de acuerdo
for w2 in ("paz", "humanitario"):
    f = bg[(bg["w1"] == "acuerdo") & (bg["w2"] == w2)]
    assert len(f) == 1, f"falta acuerdo_{w2} en la tabla NPMI"
    print(
        f"acuerdo_{w2}: n={f['n'].iloc[0]} df={f['df'].iloc[0]} npmi={f['npmi'].iloc[0]:.3f} "
        f"tipo={f['tipo'].iloc[0]} | exportado (≥{UMBRAL_NPMI}): {f['npmi'].iloc[0] >= UMBRAL_NPMI}"
    )

# 4. top-200, esquema y documentos
assert not set(PROHIBIDOS) & set(top200)
assert list(dfp.columns) == prep.SCHEMA.names
assert FAST or len(dfp) == utils.N_DOCS_UNICOS

nuevos = bg[
    (bg["w1"].isin(RESCATE_303) | bg["w2"].isin(RESCATE_303))
    & (bg["npmi"] >= UMBRAL_NPMI)
]
print(f"\nBigramas exportados con un lema rescatado: {len(nuevos)}")
print(nuevos["tipo"].value_counts().to_string())
print(nuevos.head(40)[["w1", "w2", "n", "df", "npmi", "tipo"]].round(3).to_string())
print(
    "\nOK 4.9: sustantivos rescatados fuera de locución, bigramas de acuerdo presentes, top-200 y esquema intactos"
)

## 5. Cómo lo consumen 08 y 09
```python
import utils
dfp = utils.load_corpus_procesado(columns=["id", "año", "fuente", "lemas_nom_adj", "es_breve", "es_carta"])
bg = pd.read_csv(utils.data_dir() / "bigramas_npmi.csv")
bg_topicos = set(bg.loc[bg["tipo"] == "nom_adj", "bigrama"])
```
- Para unir bigramas en S1 se usa `lemas_contenido` con `cortes_contenido`, que es el mismo flujo con el que se calcularon.
- `es_breve` y `es_carta` se usan para **estratificar o ponderar**, no para borrar. Las breves se concentran en El Tiempo 1990–2005 y excluirlas cambiaría la composición de esos años.
- `tokens_forma[:n_palabras_titulo]` es el titular y el resto es el cuerpo.